# Shape Functions on trilinear hexahedron

This notebook derives the trilinear shape functions for a 3D finite element (hexahedron).
Shape functions $N_i$ are used to interpolate values within an element based on nodal values.


In [19]:
from sympy import symbols, Eq, solve, simplify, IndexedBase, factor, diff, Matrix


In [20]:
from typst_utils import typst_print

## Symbol Definition
We define the 3D position vector $\mathbf{X} = [\xi, \eta, \zeta]^T$ and the positions of the eight nodes $\mathbf{X}_i$ for $i=0, \dots, 7$.


In [21]:
# Define symbols
X = IndexedBase('referenceposition')


## Trilinear Combination
We assume that each shape function $N_i$ is a trilinear function of the components of $\mathbf{X}$:
$N_i(\mathbf{X}) = a \cdot X[0]X[1]X[2] + b \cdot X[0]X[1] + c \cdot X[1]X[2] + d \cdot X[0]X[2] + e \cdot X[0] + f \cdot X[1] + g \cdot X[2] + h$


In [22]:
# Define symbols for coefficients
coeffs = symbols('a b c d e f g h')


## Standard Reference Hexahedron Definition
We define the coordinates of the standard reference hexahedron (nodes at $[-1, 1]^3$).
Ordering: (-1,-1,-1), (1,-1,-1), (1,1,-1), (-1,1,-1), (-1,-1,1), (1,-1,1), (1,1,1), (-1,1,1)


In [23]:
# Standard reference hex values (8 nodes)
# Ordering: (-1,-1,-1), (1,-1,-1), (1,1,-1), (-1,1,-1), (-1,-1,1), (1,-1,1), (1,1,1), (-1,1,1)
node_coords = [
    [-1, -1, -1],
    [ 1, -1, -1],
    [ 1,  1, -1],
    [-1,  1, -1],
    [-1, -1,  1],
    [ 1, -1,  1],
    [ 1,  1,  1],
    [-1,  1,  1],
]


## Solving the System
The shape functions must satisfy the Kronecker delta property: $N_i(\mathbf{X}_j) = \delta_{ij}$.
This leads to a system of eight linear equations for the eight coefficients for each $N_i$.


In [24]:
# Solve for each shape function N[i]
N = []
for i in range(8):
    # Ni(X) = a*X0*X1*X2 + b*X0*X1 + c*X1*X2 + d*X0*X2 + e*X0 + f*X1 + g*X2 + h
    eqs = [
        Eq(coeffs[0] * node_coords[j][0] * node_coords[j][1] * node_coords[j][2] + 
           coeffs[1] * node_coords[j][0] * node_coords[j][1] + 
           coeffs[2] * node_coords[j][1] * node_coords[j][2] + 
           coeffs[3] * node_coords[j][0] * node_coords[j][2] + 
           coeffs[4] * node_coords[j][0] + 
           coeffs[5] * node_coords[j][1] + 
           coeffs[6] * node_coords[j][2] + 
           coeffs[7], 1 if i == j else 0)
        for j in range(8)
    ]
    sol = solve(eqs, coeffs)
    
    Ni = (sol[coeffs[0]] * X[0] * X[1] * X[2] + 
          sol[coeffs[1]] * X[0] * X[1] + 
          sol[coeffs[2]] * X[1] * X[2] + 
          sol[coeffs[3]] * X[0] * X[2] + 
          sol[coeffs[4]] * X[0] + 
          sol[coeffs[5]] * X[1] + 
          sol[coeffs[6]] * X[2] + 
          sol[coeffs[7]])
    N.append(factor(Ni))


## Results
The expressions for the shape functions on the standard reference hexahedron are:


In [25]:
# Print results
for i in range(8):
    print(f"{typst_print(N[i])};")


-(referenceposition_x - 1) (referenceposition_y - 1) (referenceposition_z - 1)/8;
(referenceposition_x + 1) (referenceposition_y - 1) (referenceposition_z - 1)/8;
-(referenceposition_x + 1) (referenceposition_y + 1) (referenceposition_z - 1)/8;
(referenceposition_x - 1) (referenceposition_y + 1) (referenceposition_z - 1)/8;
(referenceposition_x - 1) (referenceposition_y - 1) (referenceposition_z + 1)/8;
-(referenceposition_x + 1) (referenceposition_y - 1) (referenceposition_z + 1)/8;
(referenceposition_x + 1) (referenceposition_y + 1) (referenceposition_z + 1)/8;
-(referenceposition_x - 1) (referenceposition_y + 1) (referenceposition_z + 1)/8;


## Gradients
The gradients of the shape functions $\nabla N_i = \left[ \frac{\partial N_i}{\partial X[0]}, \frac{\partial N_i}{\partial X[1]}, \frac{\partial N_i}{\partial X[2]} \right]^T$ are:


In [28]:
# Compute and print gradients
for i in range(8):
    grad_Ni = [factor(diff(N[i], X[j])) for j in range(3)]
    print(f"{typst_print(grad_Ni[0])}, {typst_print(grad_Ni[1])}, {typst_print(grad_Ni[2])};")


-(referenceposition_y - 1) (referenceposition_z - 1)/8, -(referenceposition_x - 1) (referenceposition_z - 1)/8, -(referenceposition_x - 1) (referenceposition_y - 1)/8;
(referenceposition_y - 1) (referenceposition_z - 1)/8, (referenceposition_x + 1) (referenceposition_z - 1)/8, (referenceposition_x + 1) (referenceposition_y - 1)/8;
-(referenceposition_y + 1) (referenceposition_z - 1)/8, -(referenceposition_x + 1) (referenceposition_z - 1)/8, -(referenceposition_x + 1) (referenceposition_y + 1)/8;
(referenceposition_y + 1) (referenceposition_z - 1)/8, (referenceposition_x - 1) (referenceposition_z - 1)/8, (referenceposition_x - 1) (referenceposition_y + 1)/8;
(referenceposition_y - 1) (referenceposition_z + 1)/8, (referenceposition_x - 1) (referenceposition_z + 1)/8, (referenceposition_x - 1) (referenceposition_y - 1)/8;
-(referenceposition_y - 1) (referenceposition_z + 1)/8, -(referenceposition_x + 1) (referenceposition_z + 1)/8, -(referenceposition_x + 1) (referenceposition_y - 1)/8;
(